In [1]:
import jax
from jax import numpy as jnp
import pennylane as qml
import numpy as np
import optax
from tqdm.auto import tqdm

from typing import Callable, List, Tuple, Union

import copy

In [2]:
import sys
sys.path.append('../')

from pqcqec.noise.simple_noise import PennylaneNoisyGates
from pqcqec.circuits.modify import  pennylane_state_embedding
from pqcqec.circuits.generate import generate_random_circuit

from pqcqec.simulate.simulate import run_circuit_with_noise_model

from pqcqec.utils.quaternions_utils import quaternion_to_xzy_angles, quaternion_to_zxz_angles

from pqcqec.training.jax_loss_functions import jax_fidelity_loss, jax_pure_state_fidelity

In [3]:
class StateInputModelInterleavedQuaternionPerBlockModel:
    """A class to define the Quaternion PQC model."""
    
    def __init__(self, circuit_ops:List, num_qubits:int, noise_model:PennylaneNoisyGates,
                 pqc_blocks=1, gate_blocks=1, seed=0, pqc_type='zxz'):
        """
        Initialize the PQC model with the given parameters.
        Args:
            circuit_ops (List): List of circuit operations to be applied (circuit and its inverse).
            num_qubits (int): Number of qubits in the circuit.
            noise_model (PennylaneNoisyGates): Noise model to be applied.
            pqc_blocks (int): Number of PQC blocks.
            gate_blocks (int): Number of gates per block.
            seed (int): Random seed for parameter initialization.
        """

        self.num_qubits = num_qubits
        # self.pqc_arch = pennylane_PQC_RZRXRZ_unique
        self.circuit_ops = copy.deepcopy(circuit_ops)   
        self.pqc_blocks = pqc_blocks
        self.gate_blocks = gate_blocks
        self.seed = seed
        self.noise_model = noise_model
        # self.uncomp_circuit = circuit_ops + circuit_ops[::-1]        # self.uncomp_circuit.extend([qml.adjoint(op) for op in self.circuit_ops[::-1]])
        self.num_gates = len(self.circuit_ops)

        self.qdev_cpu = qml.device("default.qubit", wires=self.num_qubits)
        self.diff_method = "backprop"  # Use backpropagation for differentiation

        # self.pqc_gates = ['rz', 'rx', 'rz']
        # self.pqc_gates = ['rx', 'rz', 'ry']

        if pqc_type == 'zxz':
            self.pqc_gates = ['rz', 'rx', 'rz']
            self.quaternion_to_pqc_angles_fn = quaternion_to_zxz_angles
        elif pqc_type == 'xzy':
            self.pqc_gates = ['rx', 'rz', 'ry']
            self.quaternion_to_pqc_angles_fn = quaternion_to_xzy_angles

        self.num_quaternion_values = 4

        self.param_sz = (int(self.pqc_blocks * jnp.ceil(self.num_gates/self.gate_blocks)), self.num_qubits, self.num_quaternion_values)

        # Initialize unit quaternions with a moderate random rotation to avoid
        # ZXZ gimbal-lock singularities (β ≈ 0 or π) that can yield NaN gradients.
        # Sample a random axis u and an angle a ∈ [a_min, a_max], then form
        # q = [cos(a/2), u*sin(a/2)].
        key = jax.random.PRNGKey(self.seed)
        key_axis, key_angle = jax.random.split(key)
        axes = jax.random.normal(key_axis, self.param_sz[:-1] + (3,), dtype=jnp.float32)
        axes = axes / (jnp.linalg.norm(axes, axis=-1, keepdims=True) + 1e-12)
        a_min, a_max = 0.2, 0.8  # radians
        angles = jax.random.uniform(key_angle, self.param_sz[:-1] + (1,), dtype=jnp.float32,
                                    minval=a_min, maxval=a_max)
        w = jnp.cos(0.5 * angles)
        v = axes * jnp.sin(0.5 * angles)
        self.quaternions = jnp.concatenate([w, v], axis=-1).astype(jnp.float32)
        # print(self.quaternions)
        # self.pqc_params = pnp.array(init_params, requires_grad=True, dtype=jnp.float32)
        


        
        @qml.qnode(self.qdev_cpu, interface='jax', diff_method=self.diff_method)
        def model_circuit(state, pqc_params, block_idx):
            """Define the PQC model circuit."""
            # 1) Apply state embedding:
            pennylane_state_embedding(state, self.num_qubits)

            # @qml.for_loop(0, self.num_gates)
            for i, op in enumerate(self.circuit_ops):
            # def loop_body(i):
                gate, qubit, param = op
                # Apply the noisy gate:
                # if not param:
                self.noise_model.apply_gate(gate, qubit, angle=param)

                # Apply PQC to the qubit:
                if (i+1) % self.gate_blocks == 0:
                    # 2) Apply the PQC gates:
                    # print(f"Applying PQC block {i // self.gate_blocks + 1} with params: {pqc_params[i // self.gate_blocks]}")
                    # print(f'PQC Params Block Shape for {i+1} : {pqc_params.shape}')

                    current_block = i // self.gate_blocks
                    pqc_params_block = pqc_params[current_block]

                    # Apply PQC gates conditionally based on block_idx
                    # If block_idx is None, apply all blocks; otherwise only apply up to block_idx
                    # Convert None to a large number to handle the comparison
                    effective_block_idx = jnp.where(block_idx is None, 999, block_idx)
                    should_apply = current_block <= effective_block_idx
                    
                    # self.pqc_arch(self.num_qubits, pqc_params_block)
                    for qubit in range(self.num_qubits):
                        for j, pqc in enumerate(self.pqc_gates):
                            # Scale the angle by should_apply (0 or 1) to conditionally apply
                            effective_angle = should_apply * pqc_params_block[qubit, j]
                            self.noise_model.apply_gate(pqc, qubit, angle=effective_angle)

            # 3) Return the output state:
            return qml.state()
        

        self.model_circuit = model_circuit
        self.batched_model_circuit = jax.jit(jax.vmap(self.model_circuit, in_axes=(0, None, None)))

    def get_model_params(self):
        """Get the model parameters."""
        return self.quaternions

    def set_model_params(self, new_params: jnp.ndarray, block_idx=None):
        """Set the model parameters."""
        if jnp.isnan(new_params).any():
            nan_indices = jnp.argwhere(jnp.isnan(new_params))
            raise ValueError(
                f"New parameters contain NaNs. Shape: {new_params.shape}. "
                f"NaN indices: {nan_indices.tolist()}"
            )
        if block_idx is None:
            self.quaternions = new_params.astype(jnp.float32)
        else:
            self.quaternions = self.quaternions.at[block_idx].set(new_params.astype(jnp.float32))

    def get_pqc_params(self):
        return self.get_pqc_params_from_all_quaternions()

    def get_pqc_params_from_block_quaternions(self, quaternions):
        """Convert quaternions to PQC parameters."""
        angles = jax.vmap(self.quaternion_to_pqc_angles_fn)(quaternions)
        return angles

    def get_pqc_params_from_all_quaternions(self):
        """Convert all quaternions to PQC parameters."""
        block_angles = jax.vmap(self.get_pqc_params_from_block_quaternions)(self.quaternions)
        return block_angles
    
    def run_model_batch(self, in_state, params=None, block_idx=None):
        """Run the model circuit on the BATCHED parameters and return the output state.

        Accepts either quaternion parameters of shape (blocks, qubits, 4) or none. 
        """
        if block_idx is None:
            if params is None:
                quats = self.quaternions
            else:
                quats = params
        
        else:
            quats = self.quaternions.at[block_idx].set(params)

        pqc_angles = jax.vmap(self.get_pqc_params_from_block_quaternions)(quats)
        return self.batched_model_circuit(in_state, pqc_angles, block_idx)

    def __call__(self, *args, **kwds):
        return self.run_model_batch(*args, **kwds)
    
    def __str__(self):
        return str(self.circuit_ops)

    def draw_mpl(self, in_state, params=None):
        """Draw the model circuit using matplotlib."""
 
        if params is None:
            params = self.get_pqc_params_from_all_quaternions()

        print(f"Drawing circuit with params: {params}")
        print(f"Input state: {in_state}")
        print(f'Model: {self}')

        return qml.draw_mpl(self.model_circuit, decimals=4)(in_state, params)

    def get_circuit_tokens(self):
        """Get the circuit tokens."""
        tokens = []
        for i, op in enumerate(self.circuit_ops):
        # def loop_body(i):
            tokens.append(op)
            
            if (i+1) % self.gate_blocks == 0:
                # 2) Apply the PQC gates:
                # print(f"Applying PQC block {i // self.gate_blocks + 1} with params: {pqc_params[i // self.gate_blocks]}")
                pqc_params_block = self.get_pqc_params_from_block_quaternions(self.quaternions[i // self.gate_blocks])
                # Add PQC parameters to the tokens:
                for qubit in range(self.num_qubits):
                    for j, pqc in enumerate(self.pqc_gates):
                        tokens.append((pqc, [qubit], [pqc_params_block[qubit, j].item()]))

        # 3) Return the circuit tokens with PQC params:
        return tokens



In [4]:
circops = generate_random_circuit(num_qubits=3, num_gates=4, seed=0, backend='list')
print(circops)
model = StateInputModelInterleavedQuaternionPerBlockModel(
    circuit_ops=circops,
    num_qubits=3,
    noise_model=PennylaneNoisyGates(),
    pqc_blocks=1,
    gate_blocks=1,
    seed=0
)

model.quaternions

[('cz', [2, 1], []), ('cx', [1, 2], []), ('h', [1], []), ('z', [1], [])]


Array([[[ 0.99478334,  0.06626027, -0.05981404, -0.04937589],
        [ 0.99435896, -0.07892875, -0.05870518,  0.03967694],
        [ 0.9625804 ,  0.09420384, -0.13345943,  0.21622497]],

       [[ 0.97833186, -0.17079061, -0.11641189,  0.0120669 ],
        [ 0.98610264, -0.09100285,  0.0188863 ,  0.13770775],
        [ 0.9907952 ,  0.07491736,  0.08827353, -0.07014347]],

       [[ 0.9905437 ,  0.10728709,  0.04232293, -0.07430631],
        [ 0.9596479 , -0.15598854,  0.13590011,  0.19045913],
        [ 0.99175495,  0.08542317, -0.0462985 ,  0.08355509]],

       [[ 0.92298627,  0.20839721, -0.31864733,  0.05595379],
        [ 0.9742453 ,  0.18279739, -0.11800209, -0.05921701],
        [ 0.93946046,  0.31981423,  0.11944785, -0.02941273]]],      dtype=float32)

In [5]:
model.get_pqc_params_from_all_quaternions()

Array([[[ 0.78689003,  0.17876643, -0.8860781 ],
        [-0.8914106 ,  0.19705261,  0.9711726 ],
        [ 0.8356116 ,  0.3281856 , -0.39368606]],

       [[-0.96018887,  0.4163836 ,  0.9848559 ],
        [-1.6366756 ,  0.18615295,  1.9141781 ],
        [ 2.3671792 ,  0.23207971, -2.5085335 ]],

       [[ 1.8716624 ,  0.23118015, -2.021413  ],
        [-2.091559  ,  0.4167791 ,  2.4834025 ],
        [ 1.1581748 ,  0.19463468, -0.9900725 ]],

       [[ 0.6397345 ,  0.7812004 , -0.51863766],
        [ 0.9368584 ,  0.43865982, -1.058274  ],
        [ 1.8969457 ,  0.6967966 , -1.9595416 ]]], dtype=float32)

In [6]:

def train_pqc_model_no_uncomp_per_block(model, dataloader, optimizer, schedule, main_loss_fn=jax_fidelity_loss, epochs=1):
    
    no_noise_model = PennylaneNoisyGates(x_rad=0, z_rad=0, delta_x=0, delta_z=0, seed=0)
    num_model_gates = len(model.circuit_ops)
    num_blocks = model.param_sz[0]
    num_gates_per_block = int(num_model_gates // num_blocks)


    for block_idx in range(num_blocks):

        print(f"Training block {block_idx + 1}/{num_blocks}")
        
        # Pre-compute the circuit slice for this block outside JIT
        circuit_slice = model.circuit_ops[:num_gates_per_block*(block_idx+1)]
        
        @jax.jit
        def update_step(params, opt_state, ideal_data):
            """Perform a single update step for the model parameters."""
            
            def loss_fn(p):
                measured = model(ideal_data, params=p, block_idx=block_idx)
                simulated = run_circuit_with_noise_model(circuit_slice, ideal_data, no_noise_model, model.num_qubits)
                return main_loss_fn(simulated, measured)

            loss, grads = jax.value_and_grad(loss_fn)(params)
            grads = jax.tree.map(lambda g: jnp.nan_to_num(g, nan=0.0, posinf=0.0, neginf=0.0), grads)
            updates, opt_state = optimizer.update(grads, opt_state, params)
            new_params = optax.apply_updates(params, updates)
            new_params = jax.tree.map(lambda p: jnp.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0), new_params)

            # Fidelity after parameter update
            measured = model(ideal_data, params=new_params, block_idx=block_idx)
            simulated = run_circuit_with_noise_model(circuit_slice, ideal_data, no_noise_model, model.num_qubits)

            fidelity = jax_pure_state_fidelity(simulated, measured)

            return opt_state, new_params, loss, fidelity

        
        opt_state = optimizer.init(model.get_model_params()[block_idx])

        for e in range(epochs):
            print(f"Epoch {e + 1}/{epochs}")
            data_iterator = tqdm(dataloader, desc="Training", total=len(dataloader), leave=False, unit='batch')
            
            # Initialize lists to track metrics for this epoch
            epoch_fidelities = []
            epoch_losses = []

            for i, batch in enumerate(data_iterator):

                # ideal_data = batch  # Assuming the first element is the ideal data
                # print(f'Batch Shape: {batch}')
                ideal_data = batch[0]  # Assuming the first element is the ideal data
                # print(f'Ideal Data Shape: {ideal_data.shape}')
                # print(f'Ideal Data \n: {ideal_data}')

                opt_state, params, loss, fidelity = update_step(model.get_model_params()[block_idx], opt_state, ideal_data)
                model.set_model_params(params, block_idx)

                # Track metrics
                epoch_fidelities.append(float(fidelity))
                epoch_losses.append(float(loss))

                current_lr = schedule(i)

                data_iterator.set_postfix_str(f"Fidelity (Ideal, Measured): {fidelity:.4e}, Loss: {loss:.4e}, LR: {current_lr:.4e}")
            
            # Print mean metrics at the end of each epoch
            mean_fidelity = np.mean(epoch_fidelities)
            mean_loss = np.mean(epoch_losses)
            print(f"Epoch {e+1} summary - Mean Fidelity: {mean_fidelity:.4e}, Mean Loss: {mean_loss:.4e}")




In [7]:
SEED = 0
NUM_QUBITS = 5
NUM_GATES = 20
NUM_GATE_BLOCKS = 5
NUM_DATA = 2500
NUM_TEST = 50
NOISE_DIST = {'x_rad': jnp.pi/100, 'z_rad': jnp.pi/100, 'delta_x': 0, 'delta_z': 0}
BATCH_SIZE = 25
EPOCHS = 5

In [8]:
from pqcqec.simulate.simulate import get_input_data
from pqcqec.utils.jax_utils import JAXStateDataset, JAXDataLoader
from pqcqec.circuits.modify import tokenize_qiskit_circuit

In [9]:
"""Run the full experiment with the given parameters."""

# Set random seed for reproducibility
jax_prng_keys = jax.random.split(jax.random.PRNGKey(SEED), 3).flatten() # Split gives us (3,2) shape, flatten to (6,) 
print(f"Using Seed and JAX PRNG Keys: {SEED, jax_prng_keys}")


# Generate ideal data
ideal_train_data = get_input_data(NUM_QUBITS, NUM_DATA, seed=jax_prng_keys[0])

# Generate noise
# train_noise = JAXNoise(x_rad=jnp.pi/100, z_rad=jnp.pi/100, shape=(num_data, num_gates * 2), seed=jax_prng_keys[1])
# print(noise_dist)

noise_model = PennylaneNoisyGates(**NOISE_DIST, seed=jax_prng_keys[1])


# Create dataset and dataloader
train_dataset = JAXStateDataset(ideal_train_data)
train_dataloader = JAXDataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, seed=jax_prng_keys[2])

# Generate random circuit list
qiskit_random_circuit = generate_random_circuit(
    num_qubits=NUM_QUBITS,
    num_gates=NUM_GATES,
    gate_dist=None,
    seed=SEED
)


qiskit_uncomp_circuit = qiskit_random_circuit

uncomp_circuit_ops = tokenize_qiskit_circuit(qiskit_uncomp_circuit)

# Initialize model
# model = StateInputModelInterleavedPQCModel(circuit_ops=uncomp_circuit_ops,
#                                         num_qubits=num_qubits,
#                                         noise_model=noise_model,
#                                         pqc_blocks=pqc_blocks,
#                                         gate_blocks=gate_blocks,
#                                         seed=jax_prng_keys[4])

model = StateInputModelInterleavedQuaternionPerBlockModel(circuit_ops=uncomp_circuit_ops,
                                        num_qubits=NUM_QUBITS,
                                        noise_model=noise_model,
                                        pqc_blocks=1,
                                        gate_blocks=NUM_GATE_BLOCKS,
                                        seed=jax_prng_keys[4])

model_params = model.get_model_params()
print(f"Model Parameters Shape: {model_params.shape}")
print(f"Model Parameter Count: {model_params.size}")

# Define optimizer
TOTAL_STEPS = int(NUM_DATA / BATCH_SIZE)
WARMUP_STEPS = int(0.1 * TOTAL_STEPS)
RESTART_PERIOD = int(0.25 * TOTAL_STEPS)

INIT_LR = 1e-4
PEAK_LR = 1e-2
MIN_LR = 5e-4

# 1. Warmup schedule
warmup = optax.linear_schedule(
    init_value=INIT_LR,
    end_value=PEAK_LR,
    transition_steps=WARMUP_STEPS
)

# 2. Cosine decay with restarts
def cosine_with_restart_schedule(step):
    step_in_period = step % RESTART_PERIOD
    cosine = 0.5 * (1 + jnp.cos(jnp.pi * step_in_period / RESTART_PERIOD))
    return MIN_LR + (PEAK_LR - MIN_LR) * cosine

# 3. Stitch warmup + cosine
schedule = optax.join_schedules(
    schedules=[warmup, cosine_with_restart_schedule],
    boundaries=[WARMUP_STEPS]
)

# 4. Optimizer chain
optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.scale_by_adam(eps=1e-8),
    optax.add_decayed_weights(weight_decay=1e-5),
    optax.scale_by_schedule(schedule),
    optax.scale(-1.0)
)


# Train the model
train_pqc_model_no_uncomp_per_block(model, train_dataloader, optimizer, schedule, epochs=EPOCHS)


# Test the model

# Generate test data
ideal_test_input_data = get_input_data(NUM_QUBITS, NUM_TEST, seed=jax_prng_keys[5])

print(f'Ideal Test Data Shape: {ideal_test_input_data.shape}')
print(f'Running circuit with noise model on test data...')
noisy_state = run_circuit_with_noise_model(
    uncomp_circuit_ops,
    ideal_test_input_data,
    noise_model,
    NUM_QUBITS,
    batched=True,
)

no_noise_model = PennylaneNoisyGates(x_rad=0, z_rad=0, delta_x=0, delta_z=0, seed=0)

ideal_out_state = run_circuit_with_noise_model(
    uncomp_circuit_ops,
    ideal_test_input_data,
    no_noise_model,
    NUM_QUBITS,
    batched=True,
)

print(f'Running PQC model on test data...')
pqc_state = model.run_model_batch(ideal_test_input_data)
batched_fidelity = jax.vmap(jax_pure_state_fidelity, in_axes=(0, 0))    

fidelity_ideal_noisy = batched_fidelity(ideal_out_state, noisy_state)
fidelity_ideal_pqc = batched_fidelity(ideal_out_state, pqc_state)

print(f"Fidelity (Ideal, Noisy): {jnp.mean(fidelity_ideal_noisy):.4e}")
print(f"Fidelity (Ideal, PQC): {jnp.mean(fidelity_ideal_pqc):.4e}")

Using Seed and JAX PRNG Keys: (0, Array([1797259609, 2579123966,  928981903, 3453687069, 4146024105,
       2718843009], dtype=uint32))
Model Parameters Shape: (4, 5, 4)
Model Parameter Count: 80
Training block 1/4
Epoch 1/5
Model Parameters Shape: (4, 5, 4)
Model Parameter Count: 80
Training block 1/4
Epoch 1/5


Training:   0%|          | 0/100 [00:00<?, ?batch/s]

Epoch 1 summary - Mean Fidelity: 1.1827e-03, Mean Loss: 9.9884e-01
Epoch 2/5


Training:   0%|          | 0/100 [00:00<?, ?batch/s]

Epoch 2 summary - Mean Fidelity: 1.0139e-02, Mean Loss: 9.9012e-01
Epoch 3/5


...

Training:   0%|          | 0/100 [00:00<?, ?batch/s]

Epoch 4 summary - Mean Fidelity: 2.6824e-02, Mean Loss: 9.7325e-01
Epoch 5/5


Training:   0%|          | 0/100 [00:00<?, ?batch/s]

Epoch 5 summary - Mean Fidelity: 2.6699e-02, Mean Loss: 9.7337e-01
Ideal Test Data Shape: (50, 32)
Running circuit with noise model on test data...
Running PQC model on test data...
Running PQC model on test data...


TypeError: where requires ndarray or scalar arguments, got <class 'NoneType'> at position 2.